# Day 12, Step 3 — Context Compression (agent)

**Problem:** Step 2 selects relevant sources but never checks their combined size. In a real system, task + RAG + memory can easily add up to more than the model should receive.

This notebook adds a hard character budget — but instead of always dropping sections in a fixed priority order, the **model** looks at what's actually in each section and decides which one to cut, from a fixed allowlist (`RAG` or `MEMORY`; `TASK` is never offered as an option). `TASK` is never dropped, and the old fixed priority order isn't gone — it's now the **safe default** the code falls back to if the model's answer can't be trusted.

**Graph:** `START -> assemble_sections -> decide_cut -> (agent decision) -> drop_rag_section / drop_memory_section / build_context -> ask_model -> END`

## Setup

`BUDGET` is the hard character limit.

In [45]:
from typing import TypedDict

import ollama
from langgraph.graph import END, START, StateGraph

MODEL = "gpt-oss:120b-cloud"
BUDGET = 150


## State

`cut_choice` holds the model's decision — `NONE`, `RAG`, or `MEMORY` — and is what the routing function reads.

In [46]:
class State(TypedDict):
    sections: dict[str, str]
    budget: int
    cut_choice: str
    context: str
    dropped_section: str
    answer: str


## `assemble_sections` — build the candidate sections

A sample support ticket: a question (`TASK`), a retrieved policy note (`RAG`), and an older, less relevant memory turn (`MEMORY`).

In [47]:
def assemble_sections(state: State):
    sections = {
        "TASK": "What's the status of ticket KB-118?",
        "RAG": "KB-118: Open tickets owned by network-team are reviewed every weekday morning.",
        "MEMORY": "User asked about a  ticket (#4402) three turns ago and it was already closed.",
    }
    return {"sections": sections, "budget": BUDGET, "dropped_section": "NONE"}


## `decide_cut` — the agent decision

Only calls the model when the sections are actually over budget — no need to ask when nothing needs cutting. When it does ask, it shows every non-`TASK` section and requires `RAG` or `MEMORY` back. If the reply isn't one of those, it falls back to `MEMORY` — the same section the old fixed-priority rule would have dropped first. That's the safe default: when the model's judgment can't be trusted, fall back to the deterministic rule.

In [48]:
def decide_cut(state: State):
    sections = state["sections"]
    
    total = sum(len(text) for text in sections.values())

    if total <= state["budget"]:
        return {"cut_choice": "NONE"}

    listing = "\n".join(f"{label}: {text}" for label, text in sections.items() if label != "TASK")
    prompt = (
        f"The assembled context is {total} characters, over the {state['budget']}-character budget. "
        "Choose ONE section to remove so it fits. Never choose TASK.\n"
        "Return only one of: RAG, MEMORY.\n\n"
        f"{listing}"
    )
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    choice = response["message"]["content"].strip().upper()
    if choice not in ("RAG", "MEMORY"):
        choice = "MEMORY"  # safe default: fall back to the old fixed-priority answer
    return {"cut_choice": choice}


## `route_after_cut` — where the agent boundary lives

Reads `cut_choice` and sends execution to the matching drop node, or straight to `build_context` when nothing needed cutting.

In [49]:
def route_after_cut(state: State):
    if state["cut_choice"] == "RAG":
        return "drop_rag_section"
    if state["cut_choice"] == "MEMORY":
        return "drop_memory_section"
    return "build_context"


## The two drop nodes — one per allowed choice

Each removes exactly the section the model picked, and records which one — chosen by the model instead of a fixed order.

In [50]:
def drop_rag_section(state: State):
    sections = dict(state["sections"])
    sections.pop("RAG", None)
    return {"sections": sections, "dropped_section": "RAG"}


def drop_memory_section(state: State):
    sections = dict(state["sections"])
    sections.pop("MEMORY", None)
    return {"sections": sections, "dropped_section": "MEMORY"}


## `build_context` and `ask_model`

Sections are always joined in a fixed order (`TASK`, `RAG`, `MEMORY`) regardless of which one was dropped, so provenance order stays predictable even though the drop decision wasn't fixed.

In [51]:
PRIORITY_ORDER = ["TASK", "RAG", "MEMORY"]


def build_context(state: State):
    lines = [f"{label}: {state['sections'][label]}" for label in PRIORITY_ORDER if label in state["sections"]]
    return {"context": "\n".join(lines)}


def ask_model(state: State):
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": state['context']}],
    )
    return {"answer": response["message"]["content"]}


## Build the graph

In [52]:
graph_builder = StateGraph(State)
graph_builder.add_node("assemble_sections", assemble_sections)
graph_builder.add_node("decide_cut", decide_cut)
graph_builder.add_node("drop_rag_section", drop_rag_section)
graph_builder.add_node("drop_memory_section", drop_memory_section)
graph_builder.add_node("build_context", build_context)
graph_builder.add_node("ask_model", ask_model)

graph_builder.add_edge(START, "assemble_sections")
graph_builder.add_edge("assemble_sections", "decide_cut")
graph_builder.add_conditional_edges("decide_cut", route_after_cut)
graph_builder.add_edge("drop_rag_section", "build_context") #TASK,MEMORY
graph_builder.add_edge("drop_memory_section", "build_context") #TASK,RAG
graph_builder.add_edge("build_context", "ask_model")
graph_builder.add_edge("ask_model", END)
graph = graph_builder.compile()


## Run it

The three sections combined are larger than the 150-character `BUDGET`, so `decide_cut` will call the model. Watch `Dropped section` to see which one it picked — and confirm `TASK` is never it.

In [53]:
result = graph.invoke({})
print(f"Budget: {BUDGET} characters")
print("Dropped section:", result["dropped_section"])
print("\nContext sent to the model:\n", result["context"])
print("\nAnswer:\n", result["answer"])


Budget: 150 characters
Dropped section: MEMORY

Context sent to the model:
 TASK: What's the status of ticket KB-118?
RAG: KB-118: Open tickets owned by network-team are reviewed every weekday morning.

Answer:
 **Ticket KB‑118 is currently **`Open`**.**

- It’s owned by the **network‑team**.
- The network‑team reviews all of their open tickets each weekday **morning**.  

So you can expect an update or any action on the ticket during the next morning review cycle. If you need a faster response, you could reach out directly to the network‑team lead.
